# LangChain + LangGraph — task-indexed cheatsheet


Mirrors the section structure of the **LangChain / LangGraph / LangSmith Python documentation**
(https://docs.langchain.com/oss/python). Each entry: question → canonical pattern → why + common mistake.

> **Note**: these examples need `pip install langchain langgraph langchain-openai langchain-anthropic
> langsmith langchain-mcp-adapters chromadb` plus an `OPENAI_API_KEY` (or equivalent). Cells are
> shown as reference patterns and are **not executed by the build script** — copy them into a
> live environment to run. The LangChain ecosystem moves fast; cross-reference the linked docs
> page if any API has shifted.


---
## Setup

Run this once after installing the dependencies.


### Setup — run me first


In [1]:
# Install (run once in your venv).
# !pip install langchain langgraph langchain-openai langchain-anthropic langsmith langchain-mcp-adapters chromadb

# Standard imports used across the cheatsheet.
import os
# os.environ['OPENAI_API_KEY'] = 'sk-...'
# os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'
# os.environ['LANGSMITH_API_KEY'] = 'lsv2_...'
# os.environ['LANGSMITH_TRACING'] = 'true'

# from langchain.chat_models import init_chat_model
# llm = init_chat_model('openai:gpt-4o-mini', temperature=0)

---
## 0. Framing your LLM / agent project

Before you reach for `create_react_agent`, decide what shape of solution your problem actually needs. The same brief — *"build a chatbot for our docs"* — can yield three radically different architectures with 100× cost differences.

### The four architectures, ranked by capability and cost

| Pattern | What it does | Latency / Cost | When to use | When NOT to use |
|---|---|---|---|---|
| **1. Single-shot prompt** | One LLM call. Input → output. | Fastest, cheapest. | Classification, summarisation, fixed-format extraction | When you need to look up unfamiliar facts (the model may hallucinate) |
| **2. RAG** (retrieval + generation) | Vector-search relevant docs, prepend to the prompt, generate. | One LLM call + one retrieval. | Q&A over a known corpus; answers grounded in documents | When the answer requires *reasoning across multiple steps* or external action |
| **3. ReAct agent** (tool-using) | LLM picks tools, calls them, sees results, iterates. Multiple LLM calls per query. | 2–10 LLM calls per query; harder to debug. | When the task needs to *take actions* (search, look up DB, compute) | If a single retrieval would suffice — the agent overhead is wasted |
| **4. Multi-agent** (supervisor + workers) | A coordinator routes between specialised sub-agents. | 5–50 LLM calls per query; very hard to debug. | Complex workflows with truly distinct sub-problems (research + code + summarise) | If three sub-agents end up doing nearly the same thing — collapse them |

### Concrete decision walkthrough

**Brief**: *"We want to build an internal Q&A bot for our engineering team's documentation."*

| Question | Answer | Implication |
|---|---|---|
| Does the bot need to take actions, or just answer questions? | Just answer | Skip ReAct/multi-agent; consider RAG or single-shot |
| Are answers grounded in a corpus, or general knowledge? | Corpus (engineering docs) | RAG, not single-shot |
| Will the corpus update frequently? | Yes (weekly) | Vector store needs scheduled rebuilds; consider chunk-level caching |
| What's the latency budget? | < 3s for 95% of queries | RAG fits; agent loops would blow the budget |
| Does the answer ever require multi-step reasoning over multiple docs? | Sometimes (5% of queries) | Start with RAG; add a "deep search" tool for the 5% that need it (single-tool agent) |

**Verdict**: RAG-first, with one optional tool ("deep search") for the small minority of multi-hop queries. Skip multi-agent entirely.

### Three more briefs and their right shapes

| Brief | Right pattern | Why |
|---|---|---|
| "Categorise customer-support emails into 12 buckets." | **Single-shot prompt** with structured output (Pydantic) | No external lookup needed; classification is a one-call problem |
| "Summarise weekly financial news for our analysts; cite sources." | **RAG** | Need fresh information; grounding in source docs is the value |
| "Triage incoming bug reports: search the codebase, check the issue tracker, suggest an owner." | **ReAct agent** with 3 tools | Multiple actions per ticket; tool composition is the value |
| "Migrate 10k legacy SQL queries to a new schema, preserving semantics." | **Multi-agent** (parser + translator + tester) | Genuinely distinct sub-problems; specialisation pays off |

### What this cheatsheet covers vs what it doesn't

This cheatsheet covers all four patterns (sections 5, 8, 9 below). It **doesn't** cover:

- **Cost / latency budgeting in detail** — the four-pattern table above is your starting point; production budgeting requires real load profiles.
- **Prompt engineering frameworks** (chain-of-thought, ReAct prompts, etc.) — covered implicitly via examples but not as a dedicated section.
- **Eval-driven development** — Section 13 sketches LangSmith eval but a real production eval suite is its own project.
- **Fine-tuning vs RAG** — almost always the answer is RAG. Fine-tuning is for *style/format* changes, not for *knowledge insertion*.

### Common framing mistakes at the LLM stage

- **Defaulting to multi-agent** because it's the most impressive-looking pattern. The agent overhead is rarely justified for tasks that single-shot or RAG could handle.
- **Skipping RAG and trying to fine-tune** a model on your corpus. Fine-tuning rarely teaches new facts reliably; it teaches *format*. RAG is the right answer for "the model needs to know about our docs".
- **Underestimating latency**. Agent loops with 5+ LLM calls add up to 30+ seconds. Test with realistic queries before committing to a pattern.
- **Not budgeting for cost during development**. ReAct agents during heavy iteration can rack up $50/day per developer; design for prompt caching and small-model fallbacks before you scale.


---
## 1. Quickstart — the minimal agent in 10 lines

The shortest path to a working agent: pick a model, declare tools, hand them to `create_react_agent`. ([docs](https://docs.langchain.com/oss/python/langchain/quickstart))


### How do I build a minimal tool-using agent?

`create_react_agent(model, tools)` from `langgraph.prebuilt`. Returns a runnable graph.


In [4]:
from dotenv import load_dotenv

load_dotenv()

from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain.agents import create_agent

@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"It's 18°C and cloudy in {city}."

llm = init_chat_model('openai:gpt-4o-mini', temperature=0)
agent = create_agent(llm, [get_weather])

result = agent.invoke({'messages': [{'role': 'user', 'content': "What's the weather in Bristol?"}]})
print(result['messages'][-1].content)

The weather in Bristol is currently 18°C and cloudy.


*Why init_chat_model*: provider-agnostic — switch to `'anthropic:claude-3-5-sonnet-latest'` by changing one string. *Common mistake*: forgetting that the messages list contains `(role, content)` dicts AND the agent appends the tool calls and tool returns to it. Always read the LAST message for the final answer.


### How do I stream the agent's reasoning step-by-step?

`agent.stream({...}, stream_mode='updates')`. Yields events as the agent runs.


In [11]:
for chunk in agent.stream(
    {'messages': [{'role': 'user', 'content': "What's the weather in Bristol?"}]},
    stream_mode='updates',
):
    print(chunk)   # one event per node update

{'model': {'messages': [AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 51, 'total_tokens': 67, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_d8acfe4d45', 'id': 'chatcmpl-DdEIF3RB6lPr4stGDfm8671jUXdCt', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e077d-3504-7c11-9e8c-be4b905d2be4-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'Bristol'}, 'id': 'call_9AF2MaqpsFCHqwJIgVGi1feN', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 51, 'output_tokens': 16, 'total_tokens': 67, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reason

*Stream modes*: `'values'` (full state after each step), `'updates'` (diffs only), `'messages'` (token-level). *Common mistake*: using `stream_mode='values'` on a long chain — generates huge events at every step.


---
## 2. Models — the chat model interface

The `init_chat_model` helper provides a single interface across providers. ([docs](https://docs.langchain.com/oss/python/langchain/models))


### How do I call a chat model directly?

`.invoke(messages)` for a single response; `.stream(messages)` for token-by-token.


In [17]:
from langchain.chat_models import init_chat_model

llm = init_chat_model('openai:gpt-4o-mini', temperature=0)
response = llm.invoke([
    {'role': 'system', 'content': 'You are concise.'},
    {'role': 'user', 'content': 'What is 2+2?'},
])
print(response.content)        # '4'
print(response.usage_metadata) # {'input_tokens': ..., 'output_tokens': ..., 'total_tokens': ...}

2 + 2 equals 4.
{'input_tokens': 22, 'output_tokens': 8, 'total_tokens': 30, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


*Why init_chat_model over importing ChatOpenAI directly*: provider-agnostic configuration via the model string. *Common mistake*: forgetting `temperature=0` for deterministic agent behaviour — non-zero temperature makes debugging harder.


### How do I switch providers without changing code?

Change the model string. The interface is identical.


In [ ]:
# llm = init_chat_model('openai:gpt-4o-mini')
# llm = init_chat_model('anthropic:claude-3-5-sonnet-latest')
# llm = init_chat_model('google_genai:gemini-1.5-flash')

*Why this matters*: portability between providers (cost / latency / capability tradeoffs).


### How do I get structured output (JSON-schema-validated)?

`.with_structured_output(Schema)` returns a model that produces the schema directly.


In [ ]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

class WeatherReport(BaseModel):
    temperature_c: float = Field(description='temperature in Celsius')
    summary: str = Field(description='one-sentence summary')

llm = init_chat_model('openai:gpt-4o-mini').with_structured_output(WeatherReport)
report = llm.invoke('Describe a typical UK autumn day in one sentence with temperature.')
print(report)                 # WeatherReport(temperature_c=12.0, summary='...')

*Why Pydantic schemas*: type-safe extraction; the LLM is forced to produce valid JSON matching the schema. *Common mistake*: passing a TypedDict — Pydantic models are preferred because they support runtime validation.


---
## 3. Messages — types and roles

Conversations are lists of typed messages. ([docs](https://docs.langchain.com/oss/python/langchain/messages))


### What are the message types?

`SystemMessage` (instructions), `HumanMessage` (user), `AIMessage` (model response), `ToolMessage` (tool result).


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage(content='You are a helpful assistant.'),
    HumanMessage(content="What's the weather?"),
    AIMessage(content='', tool_calls=[{'id': 't1', 'name': 'get_weather', 'args': {}}]),
    ToolMessage(content='18°C', tool_call_id='t1'),
    AIMessage(content='It is 18°C.'),
]

*Why typed messages over dicts*: the framework distinguishes them when routing tool calls and computing token usage. *Common mistake*: building a `HumanMessage` when the role should be `tool` — silently breaks tool-call wiring.


---
## 4. Tools — giving the agent capabilities

The `@tool` decorator turns a Python function into something the LLM can call. ([docs](https://docs.langchain.com/oss/python/langchain/tools))


### How do I declare a tool?

`@tool` decorator. The docstring is the tool description shown to the LLM.


In [ ]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

*Why the docstring matters*: it's what the LLM reads to decide whether to call the tool. Cryptic docstrings → wrong tool calls. *Common mistake*: forgetting type hints — without them the schema can't be generated.


### How do I declare a structured-input tool?

Annotate the function with a Pydantic input model.


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import tool

class SearchInput(BaseModel):
    query: str = Field(description='Search query')
    n_results: int = Field(default=5, description='How many results')

@tool('web_search', args_schema=SearchInput)
def web_search(query: str, n_results: int = 5) -> list[str]:
    """Search the web. Returns top n_results titles."""
    return [f'Result {i} for {query}' for i in range(n_results)]

*Why an args_schema*: gives the LLM precise field-level descriptions and constraints (`ge=`, `le=`, etc.). *Common mistake*: relying on docstring inference for complex args — explicit schemas are clearer.


### How do I handle tool errors gracefully?

Raise `ToolException`. The agent catches it and feeds the error message back as a `ToolMessage`.


In [ ]:
from langchain_core.tools import tool, ToolException

@tool
def divide(a: float, b: float) -> float:
    """Divide a by b."""
    if b == 0:
        raise ToolException('Division by zero')
    return a / b

*Why ToolException over generic exceptions*: the agent treats it as a recoverable error and tells the LLM, which can adjust. Generic exceptions abort the run.


---
## 5. Agents — the prebuilt and beyond

`create_react_agent` covers most cases. For more control, build a custom graph in section 6. ([docs](https://docs.langchain.com/oss/python/langchain/agents))


### How do I add a system prompt to the agent?

Pass a `prompt` string (or messages list) to `create_react_agent`.


In [ ]:
from langgraph.prebuilt import create_react_agent

agent = create_react_agent(
    llm, [get_weather, add],
    prompt='You are a helpful assistant. Always cite which tool you used.',
)

*When to use*: anytime you need consistent persona / output format / safety rules. *Common mistake*: putting tool descriptions in the prompt — that's the tool's docstring's job.


### How do I make the agent ask for confirmation before tool calls?

Use the `interrupt_before=['tools']` config. The graph pauses; you resume after approval.


In [ ]:
agent = create_react_agent(llm, [get_weather], interrupt_before=['tools'])

# Run, hit interrupt, inspect, then resume:
config = {'configurable': {'thread_id': '1'}}
state = agent.invoke({'messages': [{'role': 'user', 'content': '...'}]}, config)
# ... user reviews state['messages'][-1].tool_calls ...
agent.invoke(None, config)   # resumes from the interrupt

*When to use*: high-stakes tools (deletes, writes, payments). *Common mistake*: forgetting `thread_id` — the checkpointer can't resume without one.


---
## 6. LangGraph — `StateGraph` for custom flows

When the prebuilt agent isn't enough, drop down to `StateGraph`. Three building blocks: state schema, nodes, edges. ([docs](https://docs.langchain.com/oss/python/langgraph))


### How do I declare a state schema?

TypedDict (or Pydantic model). Use `Annotated[..., add_messages]` for fields that should accumulate.


In [ ]:
from typing import TypedDict, Annotated
from langgraph.graph import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]   # message list grows
    counter: int                                # plain int, gets overwritten

*Why `add_messages`*: tells LangGraph this list should be appended-to, not replaced. Without it, every node's return overwrites the messages list. *Common mistake*: declaring messages as a plain list — node returns clobber the conversation.


### How do I build and compile a graph?

`StateGraph(State)`, add nodes, add edges, set entry, compile.


In [ ]:
from langgraph.graph import StateGraph, START, END

def call_model(state: State) -> dict:
    response = llm.invoke(state['messages'])
    return {'messages': [response]}

graph = StateGraph(State)
graph.add_node('model', call_model)
graph.add_edge(START, 'model')
graph.add_edge('model', END)

app = graph.compile()
result = app.invoke({'messages': [{'role': 'user', 'content': 'hi'}]})

*Why START / END*: explicit terminals make the graph diagram clear. *Common mistake*: forgetting `compile()` — the StateGraph isn't runnable until compiled.


### How do I route conditionally between nodes?

`add_conditional_edges(source, router_fn, {output: target})`. Router returns a string key.


In [ ]:
def should_call_tools(state: State) -> str:
    last = state['messages'][-1]
    if hasattr(last, 'tool_calls') and last.tool_calls:
        return 'tools'
    return 'end'

graph.add_conditional_edges('model', should_call_tools, {'tools': 'tool_node', 'end': END})

*Why a router function*: clean separation of decision logic from graph structure. *Common mistake*: the router function MUST return a string that's a key in the mapping — typos silently break routing.


---
## 7. Memory — short-term and long-term

Short-term memory: per-thread message history via a checkpointer. Long-term memory: cross-thread store. ([short](https://docs.langchain.com/oss/python/langchain/short-term-memory), [long](https://docs.langchain.com/oss/python/langchain/long-term-memory))


### How do I add short-term (per-thread) memory?

Use a `MemorySaver` (or persistent checkpointer) when compiling.


In [ ]:
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent

memory = MemorySaver()
agent = create_react_agent(llm, [get_weather], checkpointer=memory)

config = {'configurable': {'thread_id': 'user-123'}}
agent.invoke({'messages': [{'role': 'user', 'content': "I'm in Bristol."}]}, config)
agent.invoke({'messages': [{'role': 'user', 'content': "What's the weather?"}]}, config)
# The second call remembers the first.

*When to use*: any chat-style application. *Common mistake*: forgetting `thread_id` — without it, each invocation is a new conversation.


### How do I persist memory across processes?

`PostgresSaver` or `SqliteSaver` instead of `MemorySaver`.


In [ ]:
from langgraph.checkpoint.sqlite import SqliteSaver

with SqliteSaver.from_conn_string('checkpoints.db') as memory:
    agent = create_react_agent(llm, [...], checkpointer=memory)
    # ... use as before

*When to use*: production. In-memory checkpointers vanish on restart. *Common mistake*: opening a SQLite checkpointer without the context manager — connection leaks.


### How do I add long-term (cross-thread) memory?

`InMemoryStore` (or persistent equivalent) holds key-value pairs that any thread can read/write.


In [ ]:
from langgraph.store.memory import InMemoryStore

store = InMemoryStore()
store.put(('users', 'user-123'), 'preferences', {'units': 'metric', 'lang': 'en'})

agent = create_react_agent(llm, [...], checkpointer=memory, store=store)

*When to use*: user preferences, learned facts, anything that should persist across sessions. *Common mistake*: confusing the store (long-term) with the checkpointer (short-term) — they solve different problems.


---
## 8. RAG — retrieval-augmented generation

Retrieve documents from a vector store, feed them to the LLM. ([docs](https://docs.langchain.com/oss/python/langchain/retrieval))


### How do I build a vector store and retriever?

Embed documents → store in vector DB → wrap as a retriever.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document

docs = [
    Document(page_content='LangGraph is a low-level orchestration framework.'),
    Document(page_content='LangSmith is the observability and eval platform.'),
]
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = Chroma.from_documents(docs, embeddings)
retriever = vector_store.as_retriever(search_kwargs={'k': 3})

*Why text-embedding-3-small*: cheap (~$0.02 / 1M tokens), good quality, fast. *Common mistake*: re-embedding documents on every run — embeddings are deterministic; cache them.


### How do I expose retrieval as a tool the agent can call?

Wrap retriever in a `@tool`. Agent calls it when relevant.


In [ ]:
from langchain_core.tools import tool

@tool
def search_docs(query: str) -> str:
    """Search the knowledge base for relevant documents."""
    docs = retriever.invoke(query)
    return '\n\n'.join(d.page_content for d in docs)

agent = create_react_agent(llm, [search_docs])

*Why tool over chain*: the agent decides WHEN to retrieve. A naive RAG chain retrieves on every input even when the query is conversational. *Common mistake*: returning the full Document objects from the tool — the LLM only needs `page_content`.


### How do I split long documents before embedding?

`RecursiveCharacterTextSplitter` with sensible chunk size + overlap.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = splitter.split_documents(long_documents)

*Why overlap*: a sentence near a chunk boundary is split across two chunks; overlap means at least one chunk has the full sentence. *Common mistake*: chunk_size = 100 — too small to embed meaningfully; the embedding becomes near-useless.


---
## 9. Multi-agent orchestration

When one agent isn't enough, compose multiple via supervisor / swarm patterns. ([docs](https://docs.langchain.com/oss/python/langchain/multi-agent))


### How do I build a supervisor + worker pattern?

A supervisor LLM routes between specialised worker agents.


In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Literal, Annotated
from langgraph.graph import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    next: str

def supervisor(state: State) -> dict:
    # supervisor LLM decides which worker to call: 'researcher', 'coder', or 'finish'
    response = llm.with_structured_output(Router).invoke(state['messages'])
    return {'next': response.next}

# routes from supervisor → workers → back to supervisor
graph = StateGraph(State)
graph.add_node('supervisor', supervisor)
graph.add_node('researcher', researcher_agent)
graph.add_node('coder', coder_agent)
graph.add_conditional_edges('supervisor', lambda s: s['next'],
                              {'researcher': 'researcher', 'coder': 'coder', 'finish': END})
graph.add_edge('researcher', 'supervisor')
graph.add_edge('coder', 'supervisor')
graph.add_edge(START, 'supervisor')

*When to use*: tasks with distinct sub-problems (research + code, parse + generate). *Common mistake*: building a multi-agent system when one agent + good tools would suffice — the coordination overhead rarely pays off below 4-5 specialisations.


### How do I share state between agents?

Use a single `State` schema; agents read and update fields on it.


In [ ]:
class SharedState(TypedDict):
    messages: Annotated[list, add_messages]
    research_findings: str
    code_artifact: str

def researcher_agent(state: SharedState) -> dict:
    findings = ...   # do work
    return {'research_findings': findings}

def coder_agent(state: SharedState) -> dict:
    code = generate_code_from(state['research_findings'])
    return {'code_artifact': code}

*Why one shared state*: simpler than passing data through messages; faster for large artifacts. *Common mistake*: trying to share state across separate compiled graphs — use a single StateGraph with multiple nodes instead.


---
## 10. MCP — Model Context Protocol integration

Connect to external MCP servers (file system, GitHub, etc.) and expose their tools to your agent. ([docs](https://docs.langchain.com/oss/python/langchain/mcp))


### How do I connect to an MCP server?

`MultiServerMCPClient` from `langchain_mcp_adapters`.


In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient({
    'filesystem': {
        'command': 'npx',
        'args': ['-y', '@modelcontextprotocol/server-filesystem', '/tmp'],
        'transport': 'stdio',
    },
    'github': {
        'url': 'http://localhost:8080/mcp',
        'transport': 'streamable_http',
    },
})

tools = await client.get_tools()
agent = create_react_agent(llm, tools)

*When to use*: standard external services (filesystem, GitHub, Slack) with off-the-shelf MCP servers. *Common mistake*: forgetting that tool retrieval is async (`await client.get_tools()`).


---
## 11. Streaming — token-by-token output

Three stream modes for different use cases. ([docs](https://docs.langchain.com/oss/python/langchain/streaming))


### How do I stream tokens to a UI?

`stream_mode='messages'` yields one event per LLM token chunk.


In [ ]:
async for event in agent.astream(
    {'messages': [{'role': 'user', 'content': 'Tell me a story.'}]},
    stream_mode='messages',
):
    chunk, metadata = event
    print(chunk.content, end='', flush=True)

*Why messages mode*: gives raw token chunks, ideal for UIs. *Common mistake*: using sync `.stream()` from an async context — leads to mysterious blocking. Use `astream` from async code.


### How do I stream node-level updates (without tokens)?

`stream_mode='updates'` yields one event per node completion.


In [ ]:
for event in agent.stream(
    {'messages': [{'role': 'user', 'content': 'What is 2+2?'}]},
    stream_mode='updates',
):
    print(event)   # {'agent': {'messages': [...]}} or {'tools': {'messages': [...]}}

*When to use*: showing progress ("calling tool X...", "thinking..."). *Common mistake*: confusing 'updates' (deltas) with 'values' (full state) — values is much heavier.


---
## 12. Observability — tracing with LangSmith

Set environment variables; everything is traced automatically. Find traces at https://smith.langchain.com. ([docs](https://docs.langchain.com/oss/python/langchain/langsmith-observability))


### How do I enable tracing?

Set `LANGSMITH_TRACING=true` and `LANGSMITH_API_KEY=...`. That's it.


In [ ]:
import os
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_API_KEY'] = 'lsv2_...'
os.environ['LANGSMITH_PROJECT'] = 'my-agent'   # optional, organises traces

# Anything you do with langchain / langgraph after this is traced automatically.
agent.invoke({'messages': [{'role': 'user', 'content': 'hi'}]})

*Why automatic tracing*: zero code changes; spans are named after your nodes / tools. *Common mistake*: forgetting LANGSMITH_PROJECT — all your traces land in 'default' and get hard to find.


### How do I add a custom tag / metadata to a run?

Pass a `config` dict with `tags=` and `metadata=`.


In [ ]:
agent.invoke(
    {'messages': [{'role': 'user', 'content': 'hi'}]},
    config={
        'tags': ['prod', 'experiment-v3'],
        'metadata': {'user_id': 'u-123', 'session_id': 's-456'},
    },
)

*When to use*: filtering production traces by user / experiment / version when debugging. *Common mistake*: putting PII in metadata — assume LangSmith may be shared / audited.


---
## 13. Evaluation — datasets, evaluators, regression suites

Capture golden examples in a dataset, run your agent against them, score with built-in or LLM-as-judge evaluators. ([docs](https://docs.langchain.com/oss/python/langsmith))


### How do I create an eval dataset?

`langsmith.Client().create_dataset(...)` then add examples.


In [ ]:
from langsmith import Client

client = Client()
dataset = client.create_dataset('weather-agent-v1', description='Common weather questions')
client.create_examples(
    inputs=[{'question': "What's the weather in Bristol?"},
            {'question': "What about Tokyo?"}],
    outputs=[{'has_temperature': True, 'has_city': True}] * 2,
    dataset_id=dataset.id,
)

*Why a versioned dataset*: regression suite. Every change to the agent is run against the same set, scores compared. *Common mistake*: putting non-deterministic outputs in `outputs` — use `outputs` for assertions, not for the exact expected text.


### How do I run an evaluator on the dataset?

`evaluate(target_fn, data=...)` runs target_fn over each example.


In [ ]:
from langsmith import evaluate

def target(inputs: dict) -> dict:
    result = agent.invoke({'messages': [{'role': 'user', 'content': inputs['question']}]})
    return {'output': result['messages'][-1].content}

def has_temperature(outputs: dict) -> bool:
    text = outputs['output'].lower()
    return any(unit in text for unit in ['°c', 'c', 'fahrenheit', 'celsius'])

results = evaluate(target, data='weather-agent-v1', evaluators=[has_temperature])

*When to use*: every PR. CI runs `evaluate`; failing evaluator = block merge. *Common mistake*: writing evaluators that are themselves LLM-judges of the same model — circular logic; results aren't reliable.


### How do I write an LLM-as-judge evaluator?

Score outputs by calling another LLM with a rubric.


In [ ]:
from langsmith import Client
from langchain.chat_models import init_chat_model

judge_llm = init_chat_model('openai:gpt-4o', temperature=0)

def is_helpful(outputs: dict, inputs: dict) -> dict:
    response = judge_llm.invoke([
        {'role': 'system', 'content': 'You are a helpfulness judge. Reply with 0 or 1.'},
        {'role': 'user', 'content': f'Q: {inputs["question"]}\nA: {outputs["output"]}\n\nIs this helpful?'},
    ])
    return {'score': 1 if '1' in response.content else 0}

*When to use*: subjective quality (helpfulness, tone, factuality). *Common mistake*: using the same model for the agent and the judge — biases the eval. Use a stronger model as judge.


---
## 14. Deployment — LangServe and FastAPI

Wrap your agent in a FastAPI app and deploy. ([LangServe docs](https://python.langchain.com/docs/langserve))


### How do I expose my agent as a FastAPI endpoint?

Wrap the agent in a route handler.


In [ ]:
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):
    message: str
    thread_id: str

@app.post('/chat')
async def chat(req: ChatRequest):
    config = {'configurable': {'thread_id': req.thread_id}}
    result = agent.invoke({'messages': [{'role': 'user', 'content': req.message}]}, config)
    return {'response': result['messages'][-1].content}

*Why FastAPI*: pydantic validation on inputs, auto-generated OpenAPI docs, async-friendly. *Common mistake*: forgetting `thread_id` — every request creates a new conversation.


### How do I stream from a FastAPI endpoint?

Return a `StreamingResponse` that yields agent stream chunks.


In [ ]:
from fastapi.responses import StreamingResponse

@app.post('/chat-stream')
async def chat_stream(req: ChatRequest):
    config = {'configurable': {'thread_id': req.thread_id}}

    async def generate():
        async for event in agent.astream(
            {'messages': [{'role': 'user', 'content': req.message}]},
            config, stream_mode='messages',
        ):
            chunk, _ = event
            if chunk.content:
                yield chunk.content

    return StreamingResponse(generate(), media_type='text/plain')

*Why StreamingResponse over WebSocket*: simpler for one-shot chat; client just reads the stream. WebSocket needed for full-duplex (e.g. interruptions). *Common mistake*: not handling disconnect — generator runs to completion even if the client is gone.
